In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df_ban_heroes = pd.read_csv('../data/ban_heroes.csv')
df_pick_heroes = pd.read_csv('../data/pick_heroes.csv')

df_draft_heroes = df_ban_heroes.merge(df_pick_heroes, on='game_id')
df_draft_heroes = df_draft_heroes.rename(columns={
  'team_1_ban_1': 'step_1', 'team_2_ban_1': 'step_2', 'team_1_ban_2': 'step_3', 'team_2_ban_2': 'step_4',
  'team_1_ban_3': 'step_5', 'team_2_ban_3': 'step_6', 'team_1_pick_1': 'step_7', 'team_2_pick_1': 'step_8',
  'team_2_pick_2': 'step_9', 'team_1_pick_2': 'step_10', 'team_1_pick_3': 'step_11', 'team_2_pick_3': 'step_12',
  'team_2_ban_4': 'step_13', 'team_1_ban_4': 'step_14', 'team_2_ban_5': 'step_15', 'team_1_ban_5': 'step_16',
  'team_2_pick_4': 'step_17', 'team_1_pick_4': 'step_18', 'team_1_pick_5': 'step_19', 'team_2_pick_5': 'step_20'
})
df_draft_heroes = df_draft_heroes[['game_id', 'step_1', 'step_2', 'step_3', 'step_4', 'step_5', 'step_6', 'step_7', 'step_8', 'step_9', 'step_10',
                                   'step_11', 'step_12', 'step_13', 'step_14','step_15', 'step_16', 'step_17', 'step_18', 'step_19', 'step_20']]
df_draft_heroes.head()

,game_id,step_1,step_2,step_3,step_4,step_5,step_6,step_7,step_8,step_9,...,step_11,step_12,step_13,step_14,step_15,step_16,step_17,step_18,step_19,step_20
0,1,fanny,chip,zhask,ling,hayabusa,zhuxin,luo yi,valentina,roger,...,moskov,arlott,x.borg,joy,hylos,nolan,chou,julian,minotaur,harith
1,2,zhuxin,chip,khufra,ling,moskov,fanny,valentina,roger,edith,...,harith,arlott,terizla,hayabusa,hylos,luo yi,vexana,minotaur,thamuz,joy
2,3,terizla,chip,zhuxin,zhask,harith,ling,valentina,khufra,moskov,...,claude,vexana,minotaur,hayabusa,carmilla,alpha,ruby,julian,hylos,nolan
3,4,terizla,chip,zhuxin,ling,harith,edith,valentina,roger,zhask,...,moskov,hylos,fanny,alpha,joy,hayabusa,guinevere,julian,x.borg,claude
4,5,zhuxin,chip,terizla,ling,zhask,edith,valentina,roger,harith,...,moskov,x.borg,fanny,luo yi,hayabusa,minotaur,guinevere,julian,paquito,novaria


In [3]:
df_game_information = pd.read_csv("../data/game_information.csv")
df_game_information.head()

,game_id,date,team_1,team_2,game_number,team_winner,patch_version
0,1,9-Aug-2024,tlid,fnoc,1,fnoc,1.9.06
1,2,9-Aug-2024,fnoc,tlid,2,fnoc,1.9.06
2,3,9-Aug-2024,dewa,evos,1,dewa,1.9.06
3,4,9-Aug-2024,dewa,evos,2,evos,1.9.06
4,5,9-Aug-2024,dewa,evos,3,dewa,1.9.06


In [64]:
STEP_AS_LABEL = 20

### Picked and Banned

In [65]:
def picked_banned_feature(df_draft_steps: pd.DataFrame, step_as_label: int):
  with open('../data/heroes2id.json', 'r') as f:
    heroes2id = json.load(f)
  total_heroes = len(heroes2id)
  ban_steps = [1, 2, 3, 4, 5, 6, 13, 14, 15, 16]
  team1_steps = [1, 3, 5, 7, 10, 11, 14, 16, 18, 19]

  banned_team1, banned_team2 = [], []
  picked_team1, picked_team2 = [], []
  labels = []

  for _, row in df_draft_steps.iterrows():
    is_banned_team1, is_banned_team2 = np.zeros(total_heroes), np.zeros(total_heroes)
    is_picked_team1, is_picked_team2 = np.zeros(total_heroes), np.zeros(total_heroes)
    
    for i in range(1, step_as_label):
      hero = row[f'step_{i}']
      hero_id = heroes2id[hero]
      if i in ban_steps:
        if i in team1_steps:
          is_banned_team1[hero_id] = 1
        else:
          is_banned_team2[hero_id] = 1
      else:
        if i in team1_steps:
          is_picked_team1[hero_id] = 1
        else:
          is_picked_team2[hero_id] = 1

    label = row[f'step_{step_as_label}']
    labels.append(heroes2id[label])
    banned_team1.append(is_banned_team1)
    banned_team2.append(is_banned_team2)
    picked_team1.append(is_picked_team1)
    picked_team2.append(is_picked_team2)
  
  return np.array(banned_team1), np.array(banned_team2), np.array(picked_team1), np.array(picked_team2), np.array(labels)

In [66]:
feat_banned_team1, feat_banned_team2, feat_picked_team1, feat_picked_team2, label = picked_banned_feature(df_draft_heroes, STEP_AS_LABEL)

print('Feature Banned Team1:', feat_banned_team1.shape)
print('Feature Banned Team2:', feat_banned_team2.shape)
print('Feature Picked Team1:', feat_picked_team1.shape)
print('Feature Picked Team2:', feat_picked_team2.shape)
print('Label:', label.shape)

Feature Banned Team1: (211, 128)
Feature Banned Team2: (211, 128)
Feature Picked Team1: (211, 128)
Feature Picked Team2: (211, 128)
Label: (211,)


### Pick Rate and Ban Rate

In [67]:
def pick_ban_rate(df_draft_steps: pd.DataFrame):
  with open('../data/heroes2id.json', 'r') as f:
    heroes2id = json.load(f)
  total_heroes = len(heroes2id)
  ban_steps = [1, 2, 3, 4, 5, 6, 13, 14, 15, 16]

  pick_count, ban_count = np.zeros(total_heroes), np.zeros(total_heroes)
  pickrate, banrate = [], []

  for _, row in df_draft_steps.iterrows():
    pickrate_game = pick_count / (row['game_id'] - 1) if row['game_id'] > 1 else pick_count
    banrate_game = ban_count / (row['game_id'] - 1) if row['game_id'] > 1 else ban_count
    pickrate.append(pickrate_game.copy())
    banrate.append(banrate_game.copy())

    for i in range(1, 21):
      hero = row[f'step_{i}']
      hero_id = heroes2id[hero]
      if i in ban_steps:
        ban_count[hero_id] += 1
      else:
        pick_count[hero_id] += 1

  return np.array(pickrate), np.array(banrate)


In [68]:
feat_pickrate, feat_banrate = pick_ban_rate(df_draft_heroes)

print('Feature Pick Rate:', feat_pickrate.shape)
print('Feature Ban Rate:', feat_banrate.shape)

Feature Pick Rate: (211, 128)
Feature Ban Rate: (211, 128)


### Role Picked

In [69]:
def role_picked(df_draft_steps: pd.DataFrame, step_as_label: int):
  with open('../data/heroes2role.json', 'r') as f:
    heroesrole = json.load(f)
  role2id = {'goldlane': 0, 'jungle': 1, 'roam': 2, 'midlane': 3, 'explane': 4}
  ban_steps = [1, 2, 3, 4, 5, 6, 13, 14, 15, 16]
  team1_steps = [1, 3, 5, 7, 10, 11, 14, 16, 18, 19]

  role_picked_team1, role_picked_team2 = [], []

  for _, row in df_draft_steps.iterrows():
    role_team1, role_team2 = np.zeros(5), np.zeros(5)

    if step_as_label > 7:
      for i in range(7, step_as_label):
        if i not in ban_steps:
          hero = row[f'step_{i}']
          roles = heroesrole[hero]
          roles_id = [role2id[role] for role in roles]
          if i in team1_steps:
            for role_id in roles_id: role_team1[role_id] = 1
          else:
            for role_id in roles_id: role_team2[role_id] = 1
    
    role_picked_team1.append(role_team1)
    role_picked_team2.append(role_team2)
  
  return np.array(role_picked_team1), np.array(role_picked_team2)

In [70]:
feat_role_team1, feat_role_team2 = role_picked(df_draft_heroes, STEP_AS_LABEL)

print('Feature Role Picked Team1:', feat_role_team1.shape)
print('Feature Role Picked Team2:', feat_role_team2.shape)

Feature Role Picked Team1: (211, 5)
Feature Role Picked Team2: (211, 5)


### Patch Version

In [71]:
with open("../data/patch_version2id.json", 'r') as f:
  patch_version2id = json.load(f)

patch_version = df_game_information['patch_version'].to_numpy()
feat_patch_version = np.array([patch_version2id[pv] for pv in patch_version]).reshape(-1, 1)

print("Feature Patch Version:", feat_patch_version.shape)

Feature Patch Version: (211, 1)


### Team

In [72]:
with open("../data/team2id.json", 'r') as f:
  team2id = json.load(f)

team1 = df_game_information['team_1'].to_numpy()
feat_team1 = np.array([team2id[t] for t in team1]).reshape(-1, 1)
team2 = df_game_information['team_2'].to_numpy()
feat_team2 = np.array([team2id[t] for t in team2]).reshape(-1, 1)

print("Feature Team 1:", feat_team1.shape)
print("Feature Team 2:", feat_team2.shape)

Feature Team 1: (211, 1)
Feature Team 2: (211, 1)


In [73]:
features = np.concatenate([
  feat_patch_version, feat_team1, feat_team2, feat_pickrate, feat_banrate, feat_role_team1,
  feat_role_team2, feat_banned_team1, feat_banned_team2, feat_picked_team1, feat_picked_team2
], axis=1, dtype=np.float16)

print(features.shape)

np.savetxt(f"../data/features/step{STEP_AS_LABEL}_features.csv", features, delimiter=",", fmt="%.4f")
np.savetxt(f"../data/features/step{STEP_AS_LABEL}_label.csv", label, delimiter=",", fmt="%.4f")

(211, 781)
